In [ ]:
# Imports and configuration
import os
import math
import requests
import pandas as pd
import plotly.graph_objects as go
from IPython.display import display, HTML

# Backend base URL (set via CASE_MGMT_BACKEND_URL)
backendUrl = os.getenv('CASE_MGMT_BACKEND_URL', 'http://localhost:3000')

# Fallback account ID (used when no account specified or when fetch returns 404)
FALLBACK_ACCOUNT_ID = 'cdtrAcct_3a1f3d24fb2046f2a28dc1fa506d6d69'

# Account to visualize can be provided via env var ACCOUNT_ID, otherwise fallback used
accountId = os.getenv('ACCOUNT_ID', FALLBACK_ACCOUNT_ID)

print(f'Using backend: {backendUrl}')
print(f'Using accountId: {accountId}')

In [ ]:
# Fetch network analysis data with fallback on 404
def fetch_account_network(account_id, backend_url):
    url = f"{backend_url}/api/v1/lakehouse/network-analysis/account/{account_id}"
    params = {'tenantId': 'DEFAULT'}
    try:
        resp = requests.get(url, params=params, timeout=15)
    except Exception as e:
        print('Request error:', e)
        return None

    # If not found and we're not already using the fallback, try fallback
    if resp.status_code == 404 and account_id != FALLBACK_ACCOUNT_ID:
        print(f'Network data not found for {account_id}, trying fallback: {FALLBACK_ACCOUNT_ID}')
        url = f"{backend_url}/api/v1/lakehouse/network-analysis/account/{FALLBACK_ACCOUNT_ID}"
        try:
            resp = requests.get(url, params=params, timeout=15)
        except Exception as e:
            print('Request error (fallback):', e)
            return None

    try:
        resp.raise_for_status()
    except Exception as e:
        print('HTTP error:', e)
        return None

    return resp.json()

# Example: fetch data now
data = fetch_account_network(accountId, backendUrl)
if not data:
    display(HTML(f"<div style='color:#ef4444'>No network data available for account: {accountId}</div>"))
else:
    print('Fetched network data keys:', list(data.keys()))

In [ ]:
# Normalise expected network payloads and build node/edge lists
# Actual format from backend: { 'network': { 'nodes': [...], 'edges': [...] }, 'accountDetails': {...}, 'meta': {...} }

nodes = []
edges = []

if data and isinstance(data, dict):
    network_data = data.get('network')
    if network_data and isinstance(network_data, dict):
        nodes = network_data.get('nodes', [])
        edges = network_data.get('edges', [])
    else:
        # Fallback to direct keys if structure differs
        nodes = data.get('nodes', [])
        edges = data.get('edges', [])

# Ensure simple structures
nodes = [n for n in nodes if isinstance(n, dict)]
edges = [e for e in edges if isinstance(e, dict)]

print(f'Nodes: {len(nodes)}, Edges: {len(edges)}')
if nodes:
    print('Sample node:', nodes[0])
if edges:
    print('Sample edge:', edges[0])

In [ ]:
# Build a simple layout (circular + center emphasis) and render with Plotly
def build_positions(nodes):
    positions = {}
    n = max(len(nodes), 1)
    # find center node (isCenter flag or the fallback account id)
    center_idx = 0
    for i, node in enumerate(nodes):
        if node.get('isCenter') or node.get('id') == accountId or node.get('id') == FALLBACK_ACCOUNT_ID:
            center_idx = i
            break

    # Place center at origin, others on circle
    radius = 300
    angle_step = 2 * math.pi / max(n - 1, 1)
    j = 0
    for i, node in enumerate(nodes):
        if i == center_idx:
            positions[node['id']] = (0.0, 0.0)
        else:
            angle = angle_step * j
            positions[node['id']] = (radius * math.cos(angle), radius * math.sin(angle))
            j += 1
    return positions

positions = build_positions(nodes)

# Create Plotly traces for edges (lines) and nodes (scatter)
edge_x = []
edge_y = []
for e in edges:
    s = e.get('source') or e.get('from')
    t = e.get('target') or e.get('to')
    if s in positions and t in positions:
        x0, y0 = positions[s]
        x1, y1 = positions[t]
        edge_x += [x0, x1, None]
        edge_y += [y0, y1, None]

edge_trace = go.Scatter(x=edge_x, y=edge_y, mode='lines', line=dict(width=1, color='#888'), hoverinfo='none')

# Node scatter - map types/status to marker styles
node_x = []
node_y = []
node_text = []
marker_size = []
marker_color = []

def color_for(node):
    # Prioritise status (alert/investigation) then type
    status = node.get('status', 'normal')
    t = node.get('type', 'account')
    if status in ('alert', 'flagged'):
        return '#ef4444'
    if status == 'investigation':
        return '#f59e0b'
    if t == 'counterparty':
        return '#6366f1'
    return '#60a5fa'

for n in nodes:
    nid = n.get('id')
    if nid not in positions:
        continue
    x, y = positions[nid]
    node_x.append(x)
    node_y.append(y)
    label = n.get('label') or nid
    node_text.append(f"{label} ({nid})")
    marker_size.append(30 if n.get('isCenter') else 18)
    marker_color.append(color_for(n))

node_trace = go.Scatter(
    x=node_x,
    y=node_y,
    mode='markers+text',
    text=[t.split(' ')[0] for t in node_text],
    hovertext=node_text,
    hoverinfo='text',
    marker=dict(
        showscale=False,
        color=marker_color,
        size=marker_size,
        line_width=2,
        line_color='#ffffff'
    ),
    textposition='bottom center'
)

fig = go.Figure(data=[edge_trace, node_trace], layout=go.Layout(
    title=f'Account Network: {accountId}',
    showlegend=False,
    hovermode='closest',
    margin=dict(b=20,l=5,r=5,t=40),
    xaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
    yaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
    height=650,
))

if nodes:
    fig.show()
else:
    display(HTML('<div>No nodes to display</div>'))